<a href="https://colab.research.google.com/github/samarranjit/Yield_Prediction/blob/main/30mresolution_data/TreeBasedModelsOn30mDatasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [31]:
corn_fn = r"/home/cholab/LabMembers/Samar/Yield_Prediction/data/monthlyTimeseriesData/2corn_monthly_timeseries_imputed_recalculated.parquet"

wheat_fn = r"/home/cholab/LabMembers/Samar/Yield_Prediction/data/monthlyTimeseriesData/2wheat_monthly_timeseries_imputed_recalculated.parquet"
soy_fn = r"/home/cholab/LabMembers/Samar/Yield_Prediction/data/monthlyTimeseriesData/2soybeans_monthly_timeseries_imputed_recalculated.parquet"
CROP = "wheat"   # "soy" | "corn" | "wheat"

fn_map = {"soy": soy_fn, "corn": corn_fn, "wheat": wheat_fn}
data_path = fn_map[CROP]

df = pd.read_parquet(data_path)
print("Loaded:", data_path)
print("Shape:", df.shape)
df.head()

Loaded: /home/cholab/LabMembers/Samar/Yield_Prediction/data/monthlyTimeseriesData/2wheat_monthly_timeseries_imputed_recalculated.parquet
Shape: (87660, 70)


,field,year,row,col,yield_mean,yield_std,yield_count,lon,lat,grid_crs,...,EVI,GI,NDWI,Tr_SWIR1,Tr_SWIR2,LST_K,PPT_mm,PPT_obsCount,HLS_obsCount,LST_obsCount
0,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.211280,2.230067,-0.093518,0.705636,1.389632,292.362000,68.996002,31.0,3.0,2.0
1,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.304914,2.919337,0.025263,1.014456,2.190767,292.362000,21.474001,28.0,6.0,2.0
2,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.380398,3.539373,0.118324,1.244815,2.836716,292.362000,89.545998,31.0,3.0,1.0
3,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.426225,3.676572,0.180334,1.349838,3.071973,305.155670,92.737999,30.0,1.0,1.0
4,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.440394,3.891422,0.172018,1.228205,2.787438,306.281921,172.608994,31.0,7.0,2.0


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87660 entries, 0 to 87659
Data columns (total 70 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   field               87660 non-null  object 
 1   year                87660 non-null  int64  
 2   row                 87660 non-null  int64  
 3   col                 87660 non-null  int64  
 4   yield_mean          87660 non-null  float64
 5   yield_std           85930 non-null  float64
 6   yield_count         87660 non-null  int64  
 7   lon                 87660 non-null  float64
 8   lat                 87660 non-null  float64
 9   grid_crs            87660 non-null  object 
 10  template_raster     87660 non-null  object 
 11  yield_meter_crs     87660 non-null  object 
 12  LST_median          87660 non-null  float64
 13  LST_max             87660 non-null  float64
 14  LST_range           87660 non-null  float64
 15  NDVI_mean           87660 non-null  float64
 16  EVI_

In [33]:
df.rename(columns={"x_utm": "x_m", "y_utm": "y_m"}, inplace=True)
df.head()

,field,year,row,col,yield_mean,yield_std,yield_count,lon,lat,grid_crs,...,EVI,GI,NDWI,Tr_SWIR1,Tr_SWIR2,LST_K,PPT_mm,PPT_obsCount,HLS_obsCount,LST_obsCount
0,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.211280,2.230067,-0.093518,0.705636,1.389632,292.362000,68.996002,31.0,3.0,2.0
1,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.304914,2.919337,0.025263,1.014456,2.190767,292.362000,21.474001,28.0,6.0,2.0
2,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.380398,3.539373,0.118324,1.244815,2.836716,292.362000,89.545998,31.0,3.0,1.0
3,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.426225,3.676572,0.180334,1.349838,3.071973,305.155670,92.737999,30.0,1.0,1.0
4,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,EPSG:4326,...,0.440394,3.891422,0.172018,1.228205,2.787438,306.281921,172.608994,31.0,7.0,2.0


In [ ]:
# month_index      241080 non-null  int64
#  57  NDVI             241080 non-null  float64
#  58  EVI              241080 non-null  float64
#  59  GI               241080 non-null  float64
#  60  NDWI             241080 non-null  float64
#  61  Tr_SWIR1         241080 non-null  float64
#  62  Tr_SWIR2         241080 non-null  float64
#  63  LST_K            241080 non-null  float64
#  64  PPT_mm           241080 non-null  float64
#  65  PPT_obsCount     241080 non-null  float64
#  66  HLS_obsCount     241080 non-null  float64
#  67  LST_obsCount     241080 non-null  float64
#  68  window

In [40]:
df.drop(columns=[
    # "PPT_mm", "month",
    # "NDVI", "EVI", "GI", "NDWI", "LST_K", "Tr_SWIR1", "Tr_SWIR2", "PPT_obsCount", "HLS_obsCount", "LST_obsCount"
'grid_crs', 'template_raster', 'yield_meter_crs'
    ], inplace=True)
df.head()

,field,year,row,col,yield_mean,yield_std,yield_count,lon,lat,LST_median,...,Soil_Theta_R,Soil_Ksat,Soil_Lambda,Soil_Alpha,Soil_N,x_m,y_m,elevation,slope,aspect
0,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,291.331482,...,0.058628,0.134959,0.341773,-0.282014,1.370181,332403.946104,4.322064e+06,52.879803,3.701279,111.084793
1,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,291.331482,...,0.058628,0.134959,0.341773,-0.282014,1.370181,332403.946104,4.322064e+06,52.879803,3.701279,111.084793
2,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,291.331482,...,0.058628,0.134959,0.341773,-0.282014,1.370181,332403.946104,4.322064e+06,52.879803,3.701279,111.084793
3,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,291.331482,...,0.058628,0.134959,0.341773,-0.282014,1.370181,332403.946104,4.322064e+06,52.879803,3.701279,111.084793
4,North Farm_ND-1-B,2017,53,93,35.253355,2.538404,4,-76.936257,39.031575,291.331482,...,0.058628,0.134959,0.341773,-0.282014,1.370181,332403.946104,4.322064e+06,52.879803,3.701279,111.084793


In [41]:
df.keys()

Index(['field', 'year', 'row', 'col', 'yield_mean', 'yield_std', 'yield_count',
       'lon', 'lat', 'LST_median', 'LST_max', 'LST_range', 'NDVI_mean',
       'EVI_mean', 'GI_mean', 'NDWI_mean', 'Tr_SWIR1_mean', 'Tr_SWIR2_mean',
       'NDVI_max', 'EVI_max', 'GI_max', 'NDWI_max', 'Tr_SWIR1_max',
       'Tr_SWIR2_max', 'NDVI_range', 'EVI_range', 'GI_range', 'NDWI_range',
       'Tr_SWIR1_range', 'Tr_SWIR2_range', 'ppt_mean', 'ppt_max', 'ppt_range',
       'ppt_total', 'Soil_Sand', 'Soil_Silt', 'Soil_Clay', 'Soil_BD',
       'Soil_pH', 'Soil_OM', 'Soil_Theta_S', 'Soil_Theta_R', 'Soil_Ksat',
       'Soil_Lambda', 'Soil_Alpha', 'Soil_N', 'x_m', 'y_m', 'elevation',
       'slope', 'aspect'],
      dtype='object')

In [42]:
df_seasonal = (
    df
    .groupby(["x_m", "y_m", "year"], as_index=False)
    .mean(numeric_only=True)
)

print("Original rows:", len(df))
print("Seasonal pixel-year rows:", len(df_seasonal))

Original rows: 87660
Seasonal pixel-year rows: 8749


In [ ]:
df.keys()

Index(['field', 'year', 'row', 'col', 'yield_mean', 'yield_std', 'yield_count',
       'lon', 'lat', 'grid_crs', 'template_raster', 'yield_meter_crs',
       'LST_median', 'LST_max', 'LST_range', 'NDVI_mean', 'EVI_mean',
       'GI_mean', 'NDWI_mean', 'Tr_SWIR1_mean', 'Tr_SWIR2_mean', 'NDVI_max',
       'EVI_max', 'GI_max', 'NDWI_max', 'Tr_SWIR1_max', 'Tr_SWIR2_max',
       'NDVI_range', 'EVI_range', 'GI_range', 'NDWI_range', 'Tr_SWIR1_range',
       'Tr_SWIR2_range', 'NDVI_obsCount', 'ppt_mean', 'ppt_max', 'ppt_range',
       'ppt_total', 'Soil_Sand', 'Soil_Silt', 'Soil_Clay', 'Soil_BD',
       'Soil_pH', 'Soil_OM', 'Soil_Theta_S', 'Soil_Theta_R', 'Soil_Ksat',
       'Soil_Lambda', 'Soil_Alpha', 'Soil_N', 'x_m', 'y_m', 'elevation',
       'slope', 'aspect'],
      dtype='object')

In [43]:
df_model = df_seasonal.copy()
df_model.head()

,x_m,y_m,year,row,col,yield_mean,yield_std,yield_count,lon,lat,...,Soil_OM,Soil_Theta_S,Soil_Theta_R,Soil_Ksat,Soil_Lambda,Soil_Alpha,Soil_N,elevation,slope,aspect
0,332403.946104,4.322064e+06,2017,53.0,93.0,35.253355,2.538404,4.0,-76.936257,39.031575,...,-0.908014,0.425776,0.058628,0.134959,0.341773,-0.282014,1.370181,52.879803,3.701279,111.084793
1,332427.274521,4.322064e+06,2017,53.0,94.0,39.823887,3.597768,13.0,-76.935988,39.031575,...,-0.940171,0.430990,0.059809,0.126548,0.328477,-0.329472,1.356183,51.431164,2.618012,117.239586
2,332427.911213,4.322094e+06,2017,52.0,94.0,42.725662,2.709190,21.0,-76.935988,39.031844,...,-0.944601,0.435864,0.062402,0.125224,0.316242,-0.373594,1.340308,51.880959,2.491522,110.951508
3,332428.547908,4.322124e+06,2017,51.0,94.0,35.742679,0.358608,13.0,-76.935988,39.032114,...,-0.888167,0.429276,0.066825,0.021985,0.310395,-0.389892,1.328873,52.336819,2.405489,107.738846
4,332429.184607,4.322153e+06,2017,50.0,94.0,36.732437,1.076467,2.0,-76.935988,39.032383,...,-0.882307,0.428713,0.063118,0.122812,0.318603,-0.373196,1.341801,52.818752,2.655015,110.727654


In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except Exception as e:
    xgb_available = False
    print("XGBoost not available:", e)

YEAR_COL   = "year"
TARGET_RAW = "yield_mean"
FIXED_VAL_YEARS = [2022, 2015, 2016]

# Ensure log target exists
if "y_log" not in df_model.columns:
    df_model = df_model.copy()
    df_model["y_log"] = np.log1p(df_model[TARGET_RAW].astype(float))

# Features: numeric only, drop obvious non-features
drop_cols = {TARGET_RAW, "y_log", YEAR_COL, "row", "col", "lat", "lon"}
feature_cols = [c for c in df_model.columns
                if c not in drop_cols and pd.api.types.is_numeric_dtype(df_model[c])]

print("Num features:", len(feature_cols))


Num features: 44


In [ ]:
RF best params: {'n_estimators': 161, 'max_depth': 1, 'min_samples_split': 16, 'min_samples_leaf': 15, 'max_features': 0.8, 'bootstrap': True}

In [26]:
import numpy as np
import xgboost as xgb

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def build_preprocessor(feature_cols):
    # Fit on train only (Pipeline handles this correctly)
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    return ColumnTransformer([("num", num_pipe, feature_cols)], remainder="drop")


def make_rf_pipeline(feature_cols):
    pre = build_preprocessor(feature_cols)

    rf = RandomForestRegressor(
        n_estimators=161,
        max_depth=1,
        min_samples_split=16,
        min_samples_leaf=15,
        max_features=0.8,
        bootstrap=True
    )

    return Pipeline([("pre", pre), ("model", rf)])

    #XGB best params: {'n_estimators': 1629, 'learning_rate': 0.004670767807091202, 'max_depth': 5, 'min_child_weight': 4.819600424745501, 'subsample': 0.810291624136574, 'colsample_bytree': 0.6859053703386455, 'gamma': 1.0873018635007852, 'reg_alpha': 4.264641635924872, 'reg_lambda': 2.5028009562399314}
# {'n_estimators': 2159, 'learning_rate': 0.0031914430639582696, 'max_depth': 6, 'min_child_weight': 14.311447050028924, 'subsample': 0.7659272067418138, 'colsample_bytree': 0.9474638752529032, 'gamma': 1.467417425111204, 'reg_alpha': 1.9492126518340904, 'reg_lambda': 1.5715351921165521}
# XGB best params: {'n_estimators': 2039, 'learning_rate': 0.0020104754290378385, 'max_depth': 10, 'min_child_weight': 6.304348883791848, 'subsample': 0.6653297650398029, 'colsample_bytree': 0.6263706543395052, 'gamma': 0.338389078154123, 'reg_alpha': 0.3839632496251091, 'reg_lambda': 4.671144007141974}
def make_xgb_pipeline(feature_cols):
    pre = build_preprocessor(feature_cols)

    xgb_model = XGBRegressor(
        n_estimators=2050,
        learning_rate=0.0020104754290378385,
        random_state=42,
        n_jobs=-1,


        max_depth= 10,
        min_child_weight= 14.311447050028924,
        subsample = 0.7659272067418138,
        colsample_bytree = 0.9474638752529032,
        gamma = 1.467417425111204,
        reg_alpha= 1.9492126518340904,
        reg_lambda=  1.5715351921165521,

        # ✅ GPU
        tree_method="hist",
        device="cuda",
    )
    return Pipeline([("pre", pre), ("model", xgb_model)])


def corr_sq_raw(y_true_raw, y_pred_raw):
    y_true_raw = np.asarray(y_true_raw, dtype=float)
    y_pred_raw = np.asarray(y_pred_raw, dtype=float)
    if np.std(y_true_raw) == 0 or np.std(y_pred_raw) == 0:
        return np.nan
    r = np.corrcoef(y_true_raw, y_pred_raw)[0, 1]
    return float(r * r)


def eval_raw_metrics_from_log_model(pipe, X, y_true_raw):
    """
    pipe predicts log1p(y). We inverse-transform with expm1 and compute metrics in RAW space.
    """
    yhat_log = pipe.predict(X)
    yhat_raw = np.expm1(yhat_log)

    mae = float(mean_absolute_error(y_true_raw, yhat_raw))               # ✅ raw units
    rmse = float(np.sqrt(mean_squared_error(y_true_raw, yhat_raw)))     # ✅ raw units
    r2 = float(r2_score(y_true_raw, yhat_raw))                           # ✅ raw space
    c2 = float(corr_sq_raw(y_true_raw, yhat_raw))                        # ✅ raw space

    return {"mae": mae, "rmse": rmse, "r2": r2, "corr2": c2}


def get_splits(df, test_year):
    all_years = sorted(df[YEAR_COL].unique())

    val_years_used = [y for y in FIXED_VAL_YEARS if y != test_year]
    train_years = [y for y in all_years if (y != test_year and y not in val_years_used)]

    train_df = df[df[YEAR_COL].isin(train_years)].copy()
    val_df = df[df[YEAR_COL].isin(val_years_used)].copy()
    test_df = df[df[YEAR_COL] == test_year].copy()
    return train_df, val_df, test_df, val_years_used


def compute_split_metrics(pipe, X, y_log, y_raw):
    """
    pipe is trained on log targets. This returns metrics on RAW space after expm1.
    """
    yhat_log = pipe.predict(X)
    yhat_raw = np.expm1(yhat_log)

    out = {
        "mae": float(mean_absolute_error(y_raw, yhat_raw)),
        "rmse": float(np.sqrt(mean_squared_error(y_raw, yhat_raw))),
        "r2": float(r2_score(y_raw, yhat_raw)),
        "corr2": float(corr_sq_raw(y_raw, yhat_raw)),
    }
    return out


# ============================================================
# ✅ ADDITION: XGBoost early stopping + GPU (version-safe)
# ============================================================

def fit_xgb_pipeline_with_early_stopping(
    pipe,
    X_train, y_train_log,
    X_val, y_val_log,
    early_stopping_rounds=50
):
    """
    Fit XGBoost with early stopping using native xgb.train,
    mirroring the tuning-time implementation exactly.
    """

    pre = pipe.named_steps["pre"]
    model = pipe.named_steps["model"]

    # preprocess
    X_tr_t = pre.fit_transform(X_train)
    X_va_t = pre.transform(X_val)

    # DMatrix
    dtrain = xgb.DMatrix(X_tr_t, label=y_train_log)
    dvalid = xgb.DMatrix(X_va_t, label=y_val_log)


    # native xgb params (same as tuning)
    xgb_params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "seed": model.random_state,

        # ✅ GPU
        "tree_method": "hist",
        "device": "cuda",

        # hyperparameters
        "eta": model.learning_rate,
        "max_depth": int(model.max_depth),
        "subsample": float(model.subsample),
        "colsample_bytree": float(model.colsample_bytree),
        "lambda": float(model.reg_lambda),
    }

    booster = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=model.n_estimators,
        evals=[(dvalid, "valid")],
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=False
    )

    return booster, pre


In [45]:
def plot_learning_curve_fold(make_pipe_fn, train_df, val_df, title,
                             train_fracs=(0.1, 0.2, 0.4, 0.6, 0.8, 1.0),
                             seed=42):

    rng = np.random.default_rng(seed)

    # Keep as DataFrames
    X_val = val_df[feature_cols]
    y_val_raw = val_df[TARGET_RAW].values

    n = len(train_df)
    sizes = [max(200, int(n*f)) for f in train_fracs]  # ensure not too tiny
    sizes = sorted(set([min(s, n) for s in sizes]))

    train_mae, val_mae = [], []

    for s in sizes:
        # subsample train
        idx = rng.choice(n, size=s, replace=False)
        sub = train_df.iloc[idx]

        X_tr = sub[feature_cols]
        y_tr_log = sub["y_log"].values
        y_tr_raw = sub[TARGET_RAW].values

        pipe = make_pipe_fn(feature_cols)
        pipe.fit(X_tr, y_tr_log)

        m_tr = compute_split_metrics(pipe, X_tr, y_tr_log, y_tr_raw)["mae"]
        m_va = compute_split_metrics(pipe, X_val, val_df["y_log"].values, y_val_raw)["mae"]

        train_mae.append(m_tr)
        val_mae.append(m_va)

    plt.figure()
    plt.plot(sizes, train_mae, marker="o", label="Train MAE (raw)")
    plt.plot(sizes, val_mae, marker="o", label="Val MAE (raw)")
    plt.xlabel("Training samples")
    plt.ylabel("MAE (raw yield units)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


## Corn Loyo Results


In [30]:
rows = []
all_years = sorted(df_model["year"].unique())

for test_year in all_years:
    train_df, val_df, test_df, val_years_used = get_splits(df_model, test_year)

    # --- MINIMAL NA FIX (target only) ---
    train_df = train_df[np.isfinite(train_df["y_log"])]
    val_df   = val_df[np.isfinite(val_df["y_log"])]
    test_df  = test_df[np.isfinite(test_df["y_log"])]

    # DataFrames for X, raw targets for metrics
    X_train = train_df[feature_cols]
    y_train_raw = train_df[TARGET_RAW].values

    X_val = val_df[feature_cols]
    y_val_raw = val_df[TARGET_RAW].values

    X_test = test_df[feature_cols]
    y_test_raw = test_df[TARGET_RAW].values



    # ---------------- RF ----------------
    rf_pipe = make_rf_pipeline(feature_cols)
    rf_pipe.fit(X_train, train_df["y_log"].values)  # training target in log space

    rf_train = eval_raw_metrics_from_log_model(rf_pipe, X_train, y_train_raw)
    rf_val   = eval_raw_metrics_from_log_model(rf_pipe, X_val,   y_val_raw)  if len(val_df) else {"mae":np.nan,"rmse":np.nan,"r2":np.nan,"corr2":np.nan}
    rf_test  = eval_raw_metrics_from_log_model(rf_pipe, X_test,  y_test_raw)

    rows.append({
        "test_year": int(test_year),
        "model": "RF",
        "val_years_used": ",".join(map(str, val_years_used)),
        "train_size": int(len(train_df)),
        "val_size": int(len(val_df)),
        "test_size": int(len(test_df)),
        "train_mae": rf_train["mae"], "train_rmse": rf_train["rmse"], "train_r2": rf_train["r2"], "train_corr2": rf_train["corr2"],
        "val_mae":   rf_val["mae"],   "val_rmse":   rf_val["rmse"],   "val_r2":   rf_val["r2"],   "val_corr2":   rf_val["corr2"],
        "test_mae":  rf_test["mae"],  "test_rmse":  rf_test["rmse"],  "test_r2":  rf_test["r2"],  "test_corr2":  rf_test["corr2"],
    })

    print(f"\n=== {test_year} | RF | ValYears={val_years_used} ===")
    print(f"Train: MAE={rf_train['mae']:.3f} RMSE={rf_train['rmse']:.3f} R2={rf_train['r2']:.3f} Corr^2={rf_train['corr2']:.3f}")
    print(f"Val  : MAE={rf_val['mae']:.3f} RMSE={rf_val['rmse']:.3f} R2={rf_val['r2']:.3f} Corr^2={rf_val['corr2']:.3f}")
    print(f"Test : MAE={rf_test['mae']:.3f} RMSE={rf_test['rmse']:.3f} R2={rf_test['r2']:.3f} Corr^2={rf_test['corr2']:.3f}")

    # ---------------- XGB ----------------
    if xgb_available:
        xgb_pipe = make_xgb_pipeline(feature_cols)

        # ✅ EARLY STOPPING + GPU (same as tuning)
        booster, pre = fit_xgb_pipeline_with_early_stopping(
            xgb_pipe,
            X_train, train_df["y_log"].values,
            X_val,   val_df["y_log"].values,
            early_stopping_rounds=50
        )

        # --- predict TRAIN ---
        X_train_t = pre.transform(X_train)
        dtrain = xgb.DMatrix(X_train_t)
        yhat_train_log = booster.predict(dtrain)
        xgb_train = eval_raw_metrics_from_log_preds(y_train_raw, yhat_train_log)

        # --- predict VAL ---
        if len(val_df):
            X_val_t = pre.transform(X_val)
            dval = xgb.DMatrix(X_val_t)
            yhat_val_log = booster.predict(dval)
            xgb_val = eval_raw_metrics_from_log_preds(y_val_raw, yhat_val_log)
        else:
            xgb_val = {"mae":np.nan,"rmse":np.nan,"r2":np.nan,"corr2":np.nan}

        # --- predict TEST ---
        X_test_t = pre.transform(X_test)
        dtest = xgb.DMatrix(X_test_t)
        yhat_test_log = booster.predict(dtest)
        xgb_test = eval_raw_metrics_from_log_preds(y_test_raw, yhat_test_log)

        rows.append({
            "test_year": int(test_year),
            "model": "XGB",
            "val_years_used": ",".join(map(str, val_years_used)),
            "train_size": int(len(train_df)),
            "val_size": int(len(val_df)),
            "test_size": int(len(test_df)),
            "train_mae": xgb_train["mae"], "train_rmse": xgb_train["rmse"], "train_r2": xgb_train["r2"], "train_corr2": xgb_train["corr2"],
            "val_mae":   xgb_val["mae"],   "val_rmse":   xgb_val["rmse"],   "val_r2":   xgb_val["r2"],   "val_corr2":   xgb_val["corr2"],
            "test_mae":  xgb_test["mae"],  "test_rmse":  xgb_test["rmse"],  "test_r2":  xgb_test["r2"],  "test_corr2":  xgb_test["corr2"],
        })

        print(f"\n=== {test_year} | XGB | ValYears={val_years_used} ===")
        print(f"Train: MAE={xgb_train['mae']:.3f} RMSE={xgb_train['rmse']:.3f} R2={xgb_train['r2']:.3f} Corr^2={xgb_train['corr2']:.3f}")
        print(f"Val  : MAE={xgb_val['mae']:.3f} RMSE={xgb_val['rmse']:.3f} R2={xgb_val['r2']:.3f} Corr^2={xgb_val['corr2']:.3f}")
        print(f"Test : MAE={xgb_test['mae']:.3f} RMSE={xgb_test['rmse']:.3f} R2={xgb_test['r2']:.3f} Corr^2={xgb_test['corr2']:.3f}")

res_all = pd.DataFrame(rows).sort_values(["model", "test_year"]).reset_index(drop=True)
res_all



=== 2014 | RF | ValYears=[2023, 2019, 2024] ===
Train: MAE=16.954 RMSE=22.525 R2=0.814 Corr^2=0.835
Val  : MAE=31.355 RMSE=39.177 R2=0.409 Corr^2=0.433
Test : MAE=39.169 RMSE=45.134 R2=-0.825 Corr^2=0.208

=== 2014 | XGB | ValYears=[2023, 2019, 2024] ===
Train: MAE=15.829 RMSE=20.829 R2=0.841 Corr^2=0.851
Val  : MAE=31.140 RMSE=39.932 R2=0.386 Corr^2=0.391
Test : MAE=44.032 RMSE=51.018 R2=-1.332 Corr^2=0.327

=== 2015 | RF | ValYears=[2023, 2019, 2024] ===
Train: MAE=16.424 RMSE=21.782 R2=0.811 Corr^2=0.833
Val  : MAE=30.893 RMSE=38.780 R2=0.421 Corr^2=0.431
Test : MAE=24.587 RMSE=31.537 R2=0.527 Corr^2=0.559

=== 2015 | XGB | ValYears=[2023, 2019, 2024] ===
Train: MAE=15.878 RMSE=21.050 R2=0.823 Corr^2=0.835
Val  : MAE=32.989 RMSE=42.353 R2=0.309 Corr^2=0.330
Test : MAE=27.647 RMSE=35.321 R2=0.407 Corr^2=0.482

=== 2016 | RF | ValYears=[2023, 2019, 2024] ===
Train: MAE=16.812 RMSE=22.265 R2=0.812 Corr^2=0.834
Val  : MAE=30.921 RMSE=38.783 R2=0.421 Corr^2=0.422
Test : MAE=37.329 RMSE=

,test_year,model,val_years_used,train_size,val_size,test_size,train_mae,train_rmse,train_r2,train_corr2,val_mae,val_rmse,val_r2,val_corr2,test_mae,test_rmse,test_r2,test_corr2
0,2014,RF,"2023,2019,2024",11572,4831,1064,16.954154,22.525200,0.814164,0.834561,31.354777,39.176752,0.409075,0.432833,39.169262,45.133881,-0.825015,0.207700
1,2015,RF,"2023,2019,2024",10986,4831,1650,16.423527,21.782404,0.810862,0.833390,30.892595,38.780075,0.420981,0.431173,24.587344,31.537212,0.526969,0.559471
2,2016,RF,"2023,2019,2024",10371,4831,2265,16.812334,22.265103,0.811997,0.834263,30.921122,38.783345,0.420883,0.421797,37.328606,44.141455,0.016278,0.371933
3,2017,RF,"2023,2019,2024",11232,4831,1404,16.449368,21.778079,0.818598,0.838470,33.525303,41.616674,0.333178,0.413576,30.684139,38.978074,0.334617,0.407153
4,2018,RF,"2023,2019,2024",11903,4831,733,16.783399,22.173386,0.806856,0.829213,31.145975,38.989762,0.414703,0.433132,24.484969,30.236184,0.191194,0.526397
5,2019,RF,"2023,2024",12636,2889,1942,16.536846,21.972339,0.814130,0.835322,30.735011,39.076017,0.525433,0.534755,31.879697,39.030829,0.058028,0.227955
6,2020,RF,"2023,2019,2024",10868,4831,1768,15.963876,21.211150,0.822998,0.842605,31.461411,39.245143,0.407010,0.424086,39.317292,49.229974,0.150861,0.397770
7,2021,RF,"2023,2019,2024",10107,4831,2529,16.186605,21.695947,0.818306,0.838169,30.748580,38.352010,0.433693,0.448036,41.954306,49.307387,-0.019047,0.387740
8,2022,RF,"2023,2019,2024",11413,4831,1223,16.224259,21.631024,0.816285,0.836567,30.405003,38.493072,0.429520,0.455987,39.459483,50.208126,0.126329,0.418241
9,2023,RF,"2019,2024",12636,3055,1776,16.574853,22.007142,0.813541,0.834771,30.239220,37.890110,0.136953,0.316525,32.949693,41.143148,0.318726,0.345432


## Hyper parameter tuning

In [46]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except Exception as e:
    xgb_available = False
    print("XGBoost not available:", e)

# ---- REQUIRED: df_model, feature_cols, YEAR_COL, TARGET_RAW, y_log already exist ----
# YEAR_COL="year", TARGET_RAW="yield_mean"
# df_model["y_log"] = np.log1p(df_model[TARGET_RAW])

def corr_sq(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    r = np.corrcoef(y_true, y_pred)[0, 1]
    return float(r*r)

def eval_raw_metrics_from_log_preds(y_true_raw, y_pred_log):
    y_pred_raw = np.expm1(y_pred_log)
    return {
        "mae":  float(mean_absolute_error(y_true_raw, y_pred_raw)),
        "rmse": float(np.sqrt(mean_squared_error(y_true_raw, y_pred_raw))),
        "r2":   float(r2_score(y_true_raw, y_pred_raw)),
        "corr2": float(corr_sq(y_true_raw, y_pred_raw)),
    }

def build_preprocessor(feature_cols):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    return ColumnTransformer([("num", num_pipe, feature_cols)], remainder="drop")

def make_rf_pipeline(feature_cols, params):
    pre = build_preprocessor(feature_cols)
    rf = RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        **params
    )
    return Pipeline([("pre", pre), ("model", rf)])

def make_xgb_pipeline(feature_cols, params):
    pre = build_preprocessor(feature_cols)
    xgb = XGBRegressor(
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
        **params
    )
    return Pipeline([("pre", pre), ("model", xgb)])


In [47]:
def year_group_cv_score(df, feature_cols, params, model_kind="rf", n_splits=5, seed=42):
    """
    Returns mean CV MAE (raw) across year-group folds, plus mean RMSE/R2/Corr2 for tracking.
    """
    X = df[feature_cols]                 # keep as DataFrame
    y_log = df["y_log"].values
    y_raw = df["yield_mean"].values      # raw target for metrics
    groups = df["year"].values

    gkf = GroupKFold(n_splits=n_splits)

    fold_metrics = []
    for tr_idx, va_idx in gkf.split(X, y_log, groups=groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr_log = y_log[tr_idx]
        y_va_raw = y_raw[va_idx]

        if model_kind == "rf":
            pipe = make_rf_pipeline(feature_cols, params)
        else:
            pipe = make_xgb_pipeline(feature_cols, params)

        pipe.fit(X_tr, y_tr_log)
        yhat_va_log = pipe.predict(X_va)
        m = eval_raw_metrics_from_log_preds(y_va_raw, yhat_va_log)
        fold_metrics.append(m)

    out = {
        "mae":  float(np.mean([m["mae"] for m in fold_metrics])),
        "rmse": float(np.mean([m["rmse"] for m in fold_metrics])),
        "r2":   float(np.mean([m["r2"] for m in fold_metrics])),
        "corr2": float(np.mean([m["corr2"] for m in fold_metrics])),
    }
    return out


In [48]:
import xgboost as xgb
from sklearn.model_selection import GroupKFold

def score_params_year_cv_xgb_earlystop(df, feature_cols, params, n_splits=5, early_stopping_rounds=50):
    """
    Robust tuning scorer for XGB with:
    - GroupKFold by year
    - train on log1p(y)
    - evaluate metrics on raw y (expm1(pred))
    - early stopping via xgb.train (native API, version-safe)
    - GPU via tree_method=gpu_hist
    """
    X = df[feature_cols]
    y_log = df["y_log"].values
    y_raw = df[TARGET_RAW].values
    groups = df[YEAR_COL].values

    gkf = GroupKFold(n_splits=n_splits)
    fold_metrics = []

    for tr_idx, va_idx in gkf.split(X, y_log, groups):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr_log = y_log[tr_idx]
        y_va_log = y_log[va_idx]
        y_va_raw = y_raw[va_idx]

        # --- preprocess on train only ---
        pre = build_preprocessor(feature_cols)
        X_tr_t = pre.fit_transform(X_tr)
        X_va_t = pre.transform(X_va)

        # --- DMatrix ---
        dtrain = xgb.DMatrix(X_tr_t, label=y_tr_log)
        dvalid = xgb.DMatrix(X_va_t, label=y_va_log)

        # --- xgb.train params (native) ---
        # Map sklearn-style names to native names when needed
        xgb_params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "seed": 42,

            # ✅ GPU
            "tree_method": "hist",
            "device": "cuda",


            # from Optuna
            "eta": params["learning_rate"],
            "max_depth": int(params["max_depth"]),
            "min_child_weight": float(params["min_child_weight"]),
            "subsample": float(params["subsample"]),
            "colsample_bytree": float(params["colsample_bytree"]),
            "gamma": float(params["gamma"]),
            "alpha": float(params["reg_alpha"]),
            "lambda": float(params["reg_lambda"]),
        }

        num_boost_round = int(params.get("n_estimators", 5000))

        booster = xgb.train(
            params=xgb_params,
            dtrain=dtrain,
            num_boost_round=num_boost_round,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )

        # predict log, evaluate raw
        yhat_va_log = booster.predict(dvalid)
        m = eval_raw_metrics_from_log_preds(y_va_raw, yhat_va_log)
        fold_metrics.append(m)

    return {k: float(np.mean([m[k] for m in fold_metrics])) for k in fold_metrics[0]}


In [49]:
import optuna

def tune_rf_optuna(df, feature_cols, n_trials=50, n_splits=5):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 1400),
            "max_depth": trial.suggest_int("max_depth", 1, 40),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.8]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        }
        m = year_group_cv_score(df, feature_cols, params, model_kind="rf", n_splits=n_splits)
        # Optimize raw MAE
        trial.set_user_attr("rmse", m["rmse"])
        trial.set_user_attr("r2", m["r2"])
        trial.set_user_attr("corr2", m["corr2"])
        return m["mae"]

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_params = study.best_params
    best_mae = study.best_value
    best_meta = study.best_trial.user_attrs
    return study, best_params, best_mae, best_meta


def tune_xgb_optuna(df, feature_cols, n_trials=60, n_splits=5):
    if not xgb_available:
        raise RuntimeError("XGBoost not available in this environment.")

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 2500),
            "learning_rate": trial.suggest_float("learning_rate", 0.00001, 0.01, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        }
        m = score_params_year_cv_xgb_earlystop(
            df, feature_cols, params,
            n_splits=n_splits,
            early_stopping_rounds=50
        )
        trial.set_user_attr("rmse", m["rmse"])
        trial.set_user_attr("r2", m["r2"])
        trial.set_user_attr("corr2", m["corr2"])
        return m["mae"]

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_params = study.best_params
    best_mae = study.best_value
    best_meta = study.best_trial.user_attrs
    return study, best_params, best_mae, best_meta


In [ ]:
from sklearn.model_selection import GroupKFold

In [50]:

# (Optional but recommended) restrict tuning to the years you actually plan to train on most of the time
# e.g., use all years 2014-2024:
df_tune = df_model.copy()

target_col = "yield_mean"

# 1) drop missing targets
df_tune = df_tune.dropna(subset=[target_col])

# 2) remove sentinel values if you use them (common in rasters)
df_tune = df_tune[df_tune[target_col] != -9999]   # add others if needed

# 3) enforce positive for log (choose one approach)
df_tune = df_tune[df_tune[target_col] > 0]        # strict (safest for log)

# optional sanity check
assert df_tune[target_col].isna().sum() == 0
assert (df_tune[target_col] <= 0).sum() == 0

rf_study, rf_best_params, rf_best_mae, rf_best_meta = tune_rf_optuna(
    df_tune, feature_cols, n_trials=50, n_splits=5
)
print("\nRF best params:", rf_best_params)
print("RF CV best MAE:", rf_best_mae)
print("RF CV meta:", rf_best_meta)

if xgb_available:
    xgb_study, xgb_best_params, xgb_best_mae, xgb_best_meta = tune_xgb_optuna(
        df_tune, feature_cols, n_trials=60, n_splits=5
    )
    print("\nXGB best params:", xgb_best_params)
    print("XGB CV best MAE:", xgb_best_mae)
    print("XGB CV meta:", xgb_best_meta)


[I 2026-01-25 16:24:42,120] A new study created in memory with name: no-name-148fbc21-34c5-45fd-85ed-a8c81097c6eb


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-01-25 16:24:51,771] Trial 0 finished with value: 11.469953122673289 and parameters: {'n_estimators': 650, 'max_depth': 4, 'min_samples_split': 29, 'min_samples_leaf': 6, 'max_features': 0.3, 'bootstrap': True}. Best is trial 0 with value: 11.469953122673289.
[I 2026-01-25 16:25:01,871] Trial 1 finished with value: 11.604453776347338 and parameters: {'n_estimators': 1068, 'max_depth': 21, 'min_samples_split': 27, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 11.469953122673289.
[I 2026-01-25 16:25:09,368] Trial 2 finished with value: 11.691217524860782 and parameters: {'n_estimators': 761, 'max_depth': 35, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': False}. Best is trial 0 with value: 11.469953122673289.
[I 2026-01-25 16:25:27,152] Trial 3 finished with value: 12.534819090014635 and parameters: {'n_estimators': 909, 'max_depth': 22, 'min_samples_split': 15, 'min_samples_leaf': 4, 'max_feat

[I 2026-01-25 16:31:40,268] A new study created in memory with name: no-name-c12c2e7c-93ed-4f22-a0e9-db69ac09b3b7


[I 2026-01-25 16:31:40,258] Trial 49 finished with value: 12.661524598406166 and parameters: {'n_estimators': 1288, 'max_depth': 10, 'min_samples_split': 24, 'min_samples_leaf': 15, 'max_features': 0.8, 'bootstrap': True}. Best is trial 43 with value: 10.989138029152516.

RF best params: {'n_estimators': 394, 'max_depth': 1, 'min_samples_split': 24, 'min_samples_leaf': 15, 'max_features': 0.8, 'bootstrap': True}
RF CV best MAE: 10.989138029152516
RF CV meta: {'rmse': 14.401517881781553, 'r2': 0.017291147792120487, 'corr2': 0.18597298304655796}


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-01-25 16:31:47,658] Trial 0 finished with value: 11.821147559510568 and parameters: {'n_estimators': 877, 'learning_rate': 5.5392069434015525e-05, 'max_depth': 4, 'min_child_weight': 3.2449581722574803, 'subsample': 0.8610594662668284, 'colsample_bytree': 0.8019669624187229, 'gamma': 2.042388632115556, 'reg_alpha': 4.836074289756241, 'reg_lambda': 1.7783613807732706}. Best is trial 0 with value: 11.821147559510568.
[I 2026-01-25 16:31:52,056] Trial 1 finished with value: 11.504842236066484 and parameters: {'n_estimators': 367, 'learning_rate': 0.0008530800208298488, 'max_depth': 7, 'min_child_weight': 13.572115581349308, 'subsample': 0.9266667830476026, 'colsample_bytree': 0.6271410788473072, 'gamma': 3.891075624955789, 'reg_alpha': 0.38883641692037896, 'reg_lambda': 3.293560853782267}. Best is trial 1 with value: 11.504842236066484.
[I 2026-01-25 16:31:56,282] Trial 2 finished with value: 11.66536339573441 and parameters: {'n_estimators': 217, 'learning_rate': 0.00103727199594

In [51]:

# (Optional but recommended) restrict tuning to the years you actually plan to train on most of the time
df_tune = df_model.copy()

target_col = "yield_mean"

# 1) drop missing targets
df_tune = df_tune.dropna(subset=[target_col])

# 2) remove sentinel values if you use them (common in rasters)
df_tune = df_tune[df_tune[target_col] != -9999]   # add others if needed

# 3) enforce positive for log (choose one approach)
df_tune = df_tune[df_tune[target_col] > 0]        # strict (safest for log)

# optional sanity check
assert df_tune[target_col].isna().sum() == 0
assert (df_tune[target_col] <= 0).sum() == 0

rf_study, rf_best_params, rf_best_mae, rf_best_meta = tune_rf_optuna(
    df_tune, feature_cols, n_trials=50, n_splits=5
)
print("\nRF best params:", rf_best_params)
print("RF CV best MAE:", rf_best_mae)
print("RF CV meta:", rf_best_meta)


[I 2026-01-25 16:40:17,880] A new study created in memory with name: no-name-7952972b-bbeb-4dfb-8633-d6b538f8f5bc


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-01-25 16:40:24,145] Trial 0 finished with value: 12.240758756655213 and parameters: {'n_estimators': 380, 'max_depth': 3, 'min_samples_split': 18, 'min_samples_leaf': 4, 'max_features': 0.8, 'bootstrap': True}. Best is trial 0 with value: 12.240758756655213.
[I 2026-01-25 16:40:51,215] Trial 1 finished with value: 12.90928539568535 and parameters: {'n_estimators': 1362, 'max_depth': 17, 'min_samples_split': 26, 'min_samples_leaf': 1, 'max_features': 0.8, 'bootstrap': True}. Best is trial 0 with value: 12.240758756655213.
[I 2026-01-25 16:41:02,302] Trial 2 finished with value: 11.90936284067121 and parameters: {'n_estimators': 704, 'max_depth': 5, 'min_samples_split': 19, 'min_samples_leaf': 6, 'max_features': 0.5, 'bootstrap': True}. Best is trial 2 with value: 11.90936284067121.
[I 2026-01-25 16:41:06,469] Trial 3 finished with value: 11.571377113248952 and parameters: {'n_estimators': 368, 'max_depth': 8, 'min_samples_split': 15, 'min_samples_leaf': 14, 'max_features': 'sqrt

In [52]:

if xgb_available:
    xgb_study, xgb_best_params, xgb_best_mae, xgb_best_meta = tune_xgb_optuna(
        df_tune, feature_cols, n_trials=60, n_splits=5
    )
    print("\nXGB best params:", xgb_best_params)
    print("XGB CV best MAE:", xgb_best_mae)
    print("XGB CV meta:", xgb_best_meta)


[I 2026-01-25 16:48:44,237] A new study created in memory with name: no-name-b0ee4fdc-b03f-4fd5-9b84-6aa2792689d2


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-01-25 16:49:02,131] Trial 0 finished with value: 11.33011811574014 and parameters: {'n_estimators': 1532, 'learning_rate': 0.0007941942326899507, 'max_depth': 9, 'min_child_weight': 9.767327932084994, 'subsample': 0.8463509442430166, 'colsample_bytree': 0.6331747639339832, 'gamma': 2.2513095287255, 'reg_alpha': 3.2772440865173174, 'reg_lambda': 9.17241080939572}. Best is trial 0 with value: 11.33011811574014.
[I 2026-01-25 16:49:39,163] Trial 1 finished with value: 11.540901790141778 and parameters: {'n_estimators': 1808, 'learning_rate': 0.00016956005354444616, 'max_depth': 9, 'min_child_weight': 5.957620899404236, 'subsample': 0.7676937496430446, 'colsample_bytree': 0.7097676678059402, 'gamma': 0.4184237052301215, 'reg_alpha': 2.192711748897257, 'reg_lambda': 7.569793210068896}. Best is trial 0 with value: 11.33011811574014.
[I 2026-01-25 16:49:44,515] Trial 2 finished with value: 11.755661146354603 and parameters: {'n_estimators': 552, 'learning_rate': 0.00018839175153529428